# jobs: FashionMNIST (amort)

project = ```iP-VAE```, host = ```yoru```, device = ```any```

**Motivation**: <br>

Create jobs for MNIST amortized VAE fits.

In [1]:
# HIDE CODE


import os, sys
from IPython.display import display

# tmp & extras dir
git_dir = os.path.join(os.environ['HOME'], 'Dropbox/git')
extras_dir = os.path.join(git_dir, 'jb-progress-2025/_extras')
fig_base_dir = os.path.join(git_dir, 'jb-progress-2025/figs')
tmp_dir = os.path.join(git_dir, 'jb-progress-2025/tmp')
git_dir = os.path.join(git_dir, 'PoissonVAE')

# GitHub
sys.path.insert(0, git_dir)
from figures.fighelper import *
from main.train_vae import *

# warnings, tqdm, & style
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
from rich.jupyter import print
%matplotlib inline
set_style()

## Setup

In [2]:
from analysis.helper import job_runner_script


def divide_list(lst: list, n: int):
	k, m = divmod(len(lst), n)
	lst_divided = [
		lst[
			i * k + min(i, m):
			(i + 1) * k + min(i + 1, m)
		] for i in range(n)
	]
	return lst_divided


def _cleanup(path, host=None):
    for f in os.listdir(path):
        cond = f.endswith('.txt')
        if host is not None:
            cond = cond and host in f
        if cond:
            os.remove(pjoin(path, f))


def _name(host, gpu_i, fit_i):
    return f"{host}-cuda{gpu_i}-fit{fit_i}"

In [3]:
save_dir = pjoin(git_dir, 'scripts')
os.makedirs(save_dir, exist_ok=True)

# delete existing job runners?
_cleanup(save_dir, None)

print(sorted(os.listdir(save_dir)))

['copyfits.sh', 'fit_vae.sh', 'kill_screens.sh', 'resume_fit.sh', 'run_sessions.sh']

## yoru (```amort```)

```<conv+b|conv+b>```

In [4]:
host = 'yoru'
_cleanup(save_dir, host)

scripts = collections.defaultdict(list)

In [5]:
model_act_map = {
    'poisson': [None],
    'gaussian': [None, 'relu'],
}
seeds = np.arange(10, 10 + 5)

beta = 1.0
n_latents = 512

dataset = 'FashionMNIST'
archi = 'conv+b|conv+b'

seeds

array([10, 11, 12, 13, 14])

In [6]:
tot = 0

for model_type, latent_act_list in model_act_map.items():
    for latent_act in latent_act_list:
        for seed in seeds:
            arg = [
                f"--latent_act '{latent_act}'" if latent_act else '',
                f"--kl_beta {beta}",
                f"--n_latents {n_latents}",
                f"--comment amort",
            ]
            arg = ' '.join(filter(None, arg))
            gpu_i = tot % torch.cuda.device_count()
            kws = dict(
                device=gpu_i,
                dataset=dataset,
                archi=archi,
                model=model_type,
                args=arg,
                seed=seed,
            )
            scripts[gpu_i].append(job_runner_script(**kws))
            tot += 1

In [7]:
print(tot)

15

In [8]:
scripts = dict(scripts)
print({k: len(v) for k, v in scripts.items()})

{0: 8, 1: 7}

### Save

In [9]:
n_fits = 4

for gpu_i, scripts_list in scripts.items():
    scripts_divided = divide_list(scripts_list, n_fits)
    for fit_i, s in enumerate(scripts_divided):
        combined = ' && '.join(s)
        save_obj(
            obj=combined,
            file_name=_name(host, gpu_i, fit_i),
            save_dir=save_dir,
            mode='txt',
        )
        print(combined.replace('&& ', '&& \n'))

[PROGRESS] 'yoru-cuda0-fit0.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts


./fit_vae.sh '0' 'FashionMNIST' 'poisson' 'conv+b|conv+b' --seed 10 --kl_beta 1.0 --n_latents 512 --comment amort 
&& 
./fit_vae.sh '0' 'FashionMNIST' 'poisson' 'conv+b|conv+b' --seed 12 --kl_beta 1.0 --n_latents 512 --comment amort

[PROGRESS] 'yoru-cuda0-fit1.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts


./fit_vae.sh '0' 'FashionMNIST' 'poisson' 'conv+b|conv+b' --seed 14 --kl_beta 1.0 --n_latents 512 --comment amort 
&& 
./fit_vae.sh '0' 'FashionMNIST' 'gaussian' 'conv+b|conv+b' --seed 11 --kl_beta 1.0 --n_latents 512 --comment amort

[PROGRESS] 'yoru-cuda0-fit2.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts


...

./fit_vae.sh '1' 'FashionMNIST' 'gaussian' 'conv+b|conv+b' --seed 10 --kl_beta 1.0 --n_latents 512 --comment amort 
&& 
./fit_vae.sh '1' 'FashionMNIST' 'gaussian' 'conv+b|conv+b' --seed 12 --kl_beta 1.0 --n_latents 512 --comment amort

[PROGRESS] 'yoru-cuda1-fit2.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts


./fit_vae.sh '1' 'FashionMNIST' 'gaussian' 'conv+b|conv+b' --seed 14 --kl_beta 1.0 --n_latents 512 --comment amort 
&& 
./fit_vae.sh '1' 'FashionMNIST' 'gaussian' 'conv+b|conv+b' --seed 11 --latent_act 'relu' --kl_beta 1.0 --n_latents 
512 --comment amort

[PROGRESS] 'yoru-cuda1-fit3.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts


./fit_vae.sh '1' 'FashionMNIST' 'gaussian' 'conv+b|conv+b' --seed 13 --latent_act 'relu' --kl_beta 1.0 --n_latents 
512 --comment amort

Print one to check

In [10]:
print(combined.replace('&& ', '&& \n'))

./fit_vae.sh '1' 'FashionMNIST' 'gaussian' 'conv+b|conv+b' --seed 13 --latent_act 'relu' --kl_beta 1.0 --n_latents 
512 --comment amort